# Mealpy Timetabling

Revenue-maximizing railway timetabling solved with several metaheuristics
from the `mealpy` library, on top of the robin simulator. The discrete part
(which services are scheduled) is obtained with the conflict-avoiding heuristic,
so mealpy only optimizes the real-valued departure times.


## 0. Load Libraries


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from robin.supply.generator.entities import SupplyGenerator
from robin.supply.entities import Supply

from craft import RevenueSimulator, Solution
from craft.mealpy import MealpyTimetabling
from mealpy import FloatVar, GA, DE, PSO, SCA

## 1. Generate Supply


In [ ]:
supply_config_path = Path('../configs/supply_generator/supply_data.yaml')
generator_config_path = Path('../configs/supply_generator/config.yaml')
generator_save_path = Path('../data/results/supply_mealpy.yaml')

Path('../data/results').mkdir(parents=True, exist_ok=True)

generator = SupplyGenerator.from_yaml(
    path_config_supply=supply_config_path,
    path_config_generator=generator_config_path,
)

generator.generate(
    n_services=25,
    output_path=generator_save_path,
    seed=42,
    progress_bar=True,
    without_conflicts=False,
)

print(f'Generated {len(generator.services)} services')

## 2. Load Supply and Generate Revenue Behavior


In [ ]:
supply = Supply.from_yaml(path=str(generator_save_path))
print(f'Loaded {len(supply.services)} services')

revenue_simulator = RevenueSimulator(supply=supply)
revenue_behavior = revenue_simulator.simulate_revenue(alpha=2/3)

print(f'Revenue behavior computed for {len(revenue_behavior)} services')

## 3. Initialize Timetabling Problem


In [ ]:
timetabling = MealpyTimetabling(
    requested_services=supply.services,
    revenue_behavior=revenue_behavior,
    safe_headway=10,
    max_stop_time=10,
)

print(f'Problem initialized with {timetabling.n_services} services')
print(f'Real variables: {len(timetabling.boundaries.real)}')

## 4. Define MEALPY Problem


In [ ]:
bounds = [FloatVar(lb=lb, ub=ub) for lb, ub in timetabling.boundaries.real]

problem = {
    'obj_func': timetabling.objective_function,
    'bounds': bounds,
    'minmax': 'max',
    'verbose': False,
}

print(f'Number of real variables: {len(bounds)}')

## 5. Run GA Optimization


In [ ]:
model = GA.BaseGA(epoch=50, pop_size=20, pc=0.9, pm=0.01)

print('Starting GA optimization...')
model.solve(problem, seed=42)

print(f'Best fitness: {model.g_best.target.fitness}')

## 6. Results


In [ ]:
best_position = model.g_best.solution
best_fitness = model.g_best.target.fitness

schedule = timetabling.get_heuristic_schedule()

print(f'Best fitness: {best_fitness}')
print(f'Scheduled services: {sum(schedule)}/{len(schedule)}')

## 7. Update Supply with Solution


In [ ]:
solution = Solution(real=best_position, discrete=schedule)
updated_services = timetabling.update_supply(str(generator_save_path), solution)

print(f'Updated supply contains {len(updated_services)} services')

## 8. Convergence


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot([data.target.fitness for data in model.history.list_global_best])
plt.xlabel('Iteration')
plt.ylabel('Fitness')
plt.title('GA Convergence')
plt.grid(True)
plt.show()

## 9. Compare Algorithms


In [ ]:
algorithms = {
    'GA': GA.BaseGA(epoch=50, pop_size=20, pc=0.9, pm=0.01),
    'DE': DE.OriginalDE(epoch=50, pop_size=20, wf=0.5, cr=0.9),
    'PSO': PSO.OriginalPSO(epoch=50, pop_size=20, c1=1.5, c2=1.5, w=0.7),
    'SCA': SCA.OriginalSCA(epoch=50, pop_size=20),
}

results = {}
for name, algo in algorithms.items():
    print(f'Running {name}...')
    algo.solve(problem, seed=42)
    results[name] = algo.g_best.target.fitness
    print(f'{name} best fitness: {results[name]:.2f}')

print('\n=== Results Summary ===')
for name, fitness in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'{name}: {fitness:.2f}')